# DriftCall — Clone & Train (Colab)

> One-cell-per-step Colab notebook that clones the DriftCall repo, installs
> dependencies, runs a real GRPO training stage on the Gemma-3n-E2B base
> model, and pushes the trained LoRA back to the Hugging Face Hub.

| | |
|---|---|
| Repo | [saumilyagupta/openenv-DGXAI](https://github.com/saumilyagupta/openenv-DGXAI) |
| Branch | `main` |
| Trained adapter target | [`DGXAI/gemma-3n-e2b-driftcall-lora`](https://huggingface.co/DGXAI/gemma-3n-e2b-driftcall-lora) |
| Live Space | [`saumilyajj/driftcall`](https://huggingface.co/spaces/saumilyajj/driftcall) |
| Recommended hardware | Colab `T4` (free) — works; `A100` (Pro+) — fast |

**Before you run cell §03**, make sure your Hugging Face token has
`write` access to `DGXAI/*` or change `LORA_REPO`
in §05 to a namespace you own.


In [ ]:
# §01 — Clone the DriftCall repo
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/saumilyagupta/openenv-DGXAI.git"
REPO_BRANCH = "main"
WORKDIR = Path("/content/openenv-DGXAI")
DRIFTCALL_DIR = WORKDIR / "DRIFTCALL"

if WORKDIR.exists():
    print(f"[clone] {WORKDIR} already exists — pulling latest")
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "--all"], check=True)
    subprocess.run(
        ["git", "-C", str(WORKDIR), "checkout", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(WORKDIR), "reset", "--hard", f"origin/{REPO_BRANCH}"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
         REPO_URL, str(WORKDIR)],
        check=True,
    )

assert DRIFTCALL_DIR.exists(), f"DRIFTCALL/ not found under {WORKDIR}"
os.chdir(DRIFTCALL_DIR)
sys.path.insert(0, str(DRIFTCALL_DIR))
print(f"[clone] cwd = {Path.cwd()}")
print(f"[clone] head = " + subprocess.check_output(
    ["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True
).strip())


In [ ]:
# §02 — Install pinned dependencies
# Heavy: torch + unsloth + trl + transformers + faster-whisper + kokoro.
# Expect ~5 min on Colab T4.
import subprocess
import sys

PIP_PINS = [
    "torch>=2.5,<3.0",
    "transformers>=4.46,<5.0",
    "trl>=0.23,<0.25",
    "unsloth==2026.4.8",
    "unsloth-zoo>=2026.4.5",
    "datasets>=3.0",
    "accelerate>=1.1",
    "peft>=0.13",
    "bitsandbytes>=0.45",
    "huggingface-hub>=0.27",
    "soundfile>=0.12",
    "librosa>=0.10",
    "rapidfuzz>=3.10",
]

# Quiet install — uncomment "-v" if a wheel breaks.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade", "--no-cache-dir", *PIP_PINS],
    check=True,
)

# Verify GPU is wired.
import torch
print(f"[install] torch={torch.__version__} cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[install] device 0 = {torch.cuda.get_device_name(0)}")
else:
    print("[install] WARNING: no GPU — switch Runtime → GPU before §04")


In [ ]:
# §03 — Hugging Face authentication
# Paste a HF token with write access to the org you want to push the LoRA to.
import os
from huggingface_hub import login, whoami

# Colab way (interactive):
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    pass

if "HF_TOKEN" not in os.environ:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN (write-scope): ").strip()

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print(f"[auth] logged in as {whoami()['name']}")

# Wandb is optional and disabled by default to avoid extra prompts.
os.environ["WANDB_MODE"] = "disabled"


In [ ]:
# §04 — Train one GRPO stage
# Defaults to Stage 2 (single-drift, mixed languages) for a balanced demo.
# Increase NUM_STEPS / change STAGE for a fuller run.
import subprocess
import sys
from pathlib import Path

STAGE = 2          # 1 = warmup, 2 = single drift, 3 = compound drift
NUM_STEPS = 30     # bump to 150–200 for a real curriculum stage
NUM_GENERATIONS = 2  # G in GRPO; 8 is canonical, 2 keeps Colab T4 happy
HARDWARE = "h100"   # the script reads this for its own dtype/precision picks
OUTPUT_DIR = Path("/content/openenv-DGXAI/DRIFTCALL/checkpoints/colab/final")
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/train_driftcall_grpo.py",
    "--stage", str(STAGE),
    "--num-steps", str(NUM_STEPS),
    "--num-generations", str(NUM_GENERATIONS),
    "--hardware", HARDWARE,
    "--output-dir", str(OUTPUT_DIR),
]
print("[train] running:", " ".join(cmd))

# Stream stdout/stderr live so the Colab user sees [train] step= lines tick.
proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert proc.stdout is not None
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print(f"[train] exit_code={proc.returncode}")
assert proc.returncode == 0, f"trainer failed with exit code {proc.returncode}"


In [ ]:
# §05 — Push the trained LoRA back to the Hub
# This force-uploads the contents of OUTPUT_DIR to LORA_REPO.
# Change LORA_REPO if you don't have write access to the default org.
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo

LORA_REPO = "DGXAI/gemma-3n-e2b-driftcall-lora"
OUTPUT_DIR = Path("/content/openenv-DGXAI/DRIFTCALL/checkpoints/colab/final")

assert OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()), (
    f"no checkpoint at {OUTPUT_DIR} — did §04 finish cleanly?"
)

api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(LORA_REPO, repo_type="model", exist_ok=True, private=False,
            token=os.environ["HF_TOKEN"])

print(f"[push] uploading {OUTPUT_DIR} → https://huggingface.co/{LORA_REPO}")
api.upload_folder(
    folder_path=str(OUTPUT_DIR),
    repo_id=LORA_REPO,
    repo_type="model",
    commit_message="colab: clone-and-train run via build_colab_train_notebook.py",
)
print(f"[push] done. browse → https://huggingface.co/{LORA_REPO}")


---

### Done.

You just trained a real GRPO stage end-to-end and pushed the resulting
adapter to the Hub. The trained LoRA is live at
[`DGXAI/gemma-3n-e2b-driftcall-lora`](https://huggingface.co/DGXAI/gemma-3n-e2b-driftcall-lora) and any Space that
loads `unsloth/gemma-3n-E2B-it` + this LoRA will pick it up.

> **Want to feel the impact?** Open
> [`saumilyajj/driftcall`](https://huggingface.co/spaces/saumilyajj/driftcall), hit `/demo/`,
> and toggle between *base* and *trained* in the checkpoint radio.
